In [1]:
# PP1 Live Detection Demonstration
# Step 1: Import libraries and locate project files

from pathlib import Path
import pandas as pd
import joblib

# Find the main RA 1 project folder
current_folder = Path.cwd()

if current_folder.name == "notebooks":
    PROJECT_ROOT = current_folder.parent
else:
    PROJECT_ROOT = current_folder

# Define existing project locations
MODEL_DIR = PROJECT_ROOT / "models"
DATA_DIR = PROJECT_ROOT / "Dataset" / "processed"
DEMO_DIR = PROJECT_ROOT / "Demo outputs"

print("Project folder:", PROJECT_ROOT)
print("Models folder found:", MODEL_DIR.is_dir())
print("Processed data folder found:", DATA_DIR.is_dir())
print("Demo outputs folder found:", DEMO_DIR.is_dir())
print("Step 1 completed!")

Project folder: C:\Users\Meishaaa\OneDrive - Sri Lanka Institute of Information Technology\Desktop updates\Desktop\RA 1
Models folder found: True
Processed data folder found: True
Demo outputs folder found: True
Step 1 completed!


In [2]:
# Step 2: Load the existing trained model and preprocessor

model_path = MODEL_DIR / "detection_model.joblib"
preprocessor_path = MODEL_DIR / "detection_preprocessor.joblib"

model = joblib.load(model_path)
preprocessor = joblib.load(preprocessor_path)

print("Detection model loaded successfully!")
print("Preprocessor loaded successfully!")
print("Step 2 completed!")

Detection model loaded successfully!
Preprocessor loaded successfully!
Step 2 completed!


In [3]:
# Step 3: Load existing test data (read-only)

X_test = joblib.load(DATA_DIR / "X_test.joblib")
y_test = joblib.load(DATA_DIR / "y_test.joblib")

print("Test data loaded successfully!")
print("Number of test requests:", len(X_test))
print("Number of attack labels:", y_test.shape[1])

print("\nFirst test HTTP request:")
display(X_test.iloc[[0]])

print("Step 3 completed!")

Test data loaded successfully!
Number of test requests: 181014
Number of attack labels: 13

First test HTTP request:


,request_http_request,request_body,request_http_method,request_http_protocol,request_host,request_user_agent,request_referer,request_origin,request_cookie,request_content_type,request_accept,request_accept_language,request_accept_encoding,request_connection
568169,/,,GET,HTTP/1.1,test-site.com,Mozilla/5.0 (Windows NT 6.1; WOW64; Trident/7....,,,,,*/*,,"gzip, deflate",keep-alive


Step 3 completed!


In [4]:
# Step 4: Predict attacks for ONE test HTTP request

# Select the first request from X_test
sample = X_test.iloc[[0]]

# Convert the request into numerical features
sample_processed = preprocessor.transform(sample)

# Ask the trained model to predict attack probabilities
probabilities = model.predict_proba(sample_processed)

# Convert probabilities into 0 or 1 predictions
predictions = (probabilities >= 0.5).astype(int)

# Display the results
print("Model predictions:")
print(predictions)

print("\nPrediction probabilities:")
print(probabilities)

print("\nActual attack labels:")
print(y_test.iloc[[0]].values)

print("\nStep 4 completed!")

Model predictions:
[[1 0 0 0 0 0 0 0 0 0 0 0 0]]

Prediction probabilities:
[[9.99943728e-01 7.15978663e-06 1.15492928e-05 4.20697952e-06
  7.76251330e-06 1.70357150e-05 1.44655583e-06 6.47890019e-06
  3.57166310e-07 1.51849016e-06 6.60176864e-06 5.43108646e-06
  1.30547895e-06]]

Actual attack labels:
[[1 0 0 0 0 0 0 0 0 0 0 0 0]]

Step 4 completed!


In [5]:
# Step 5: Display readable attack names and probabilities

# Get the 13 attack label names
label_names = list(y_test.columns)

# Create a table of predictions
results_table = pd.DataFrame({
    "Attack Label": label_names,
    "Probability (%)": (probabilities[0] * 100).round(4),
    "Predicted": predictions[0],
    "Actual": y_test.iloc[0].values
})

# Check whether each label was predicted correctly
results_table["Correct"] = (
    results_table["Predicted"] == results_table["Actual"]
)

# Display all 13 attack labels
display(results_table)

# Show a simple summary
predicted_attacks = results_table.loc[
    results_table["Predicted"] == 1, "Attack Label"
].tolist()

actual_attacks = results_table.loc[
    results_table["Actual"] == 1, "Attack Label"
].tolist()

print("Predicted labels:", predicted_attacks)
print("Actual labels:", actual_attacks)
print(
    "All 13 labels matched:",
    bool(results_table["Correct"].all())
)

print("Step 5 completed!")

,Attack Label,Probability (%),Predicted,Actual,Correct
0,000 - Normal,99.9944,1,1,True
1,272 - Protocol Manipulation,0.0007,0,0,True
2,242 - Code Injection,0.0012,0,0,True
3,88 - OS Command Injection,0.0004,0,0,True
4,126 - Path Traversal,0.0008,0,0,True
5,66 - SQL Injection,0.0017,0,0,True
6,16 - Dictionary-based Password Attack,0.0001,0,0,True
7,310 - Scanning for Vulnerable Software,0.0006,0,0,True
8,153 - Input Data Manipulation,0.0000,0,0,True
9,274 - HTTP Verb Tampering,0.0002,0,0,True


Predicted labels: ['000 - Normal']
Actual labels: ['000 - Normal']
All 13 labels matched: True
Step 5 completed!


In [6]:
# Step 6: Save demonstration results separately

DEMO_DIR.mkdir(parents=True, exist_ok=True)

output_file = DEMO_DIR / "demo_request_1_predictions.csv"

results_table.to_csv(output_file, index=False)

print("Demo results saved successfully!")
print("Saved location:", output_file)
print("Step 6 completed!")

Demo results saved successfully!
Saved location: C:\Users\Meishaaa\OneDrive - Sri Lanka Institute of Information Technology\Desktop updates\Desktop\RA 1\Demo outputs\demo_request_1_predictions.csv
Step 6 completed!


In [7]:
# Step 7: Find and test ONE actual attack request

# Find requests that have at least one attack label
attack_rows = y_test[
    y_test.drop(columns=["000 - Normal"]).sum(axis=1) > 0
]

# Select the first attack request
attack_index = attack_rows.index[0]

# Get the HTTP request and its actual labels
attack_sample = X_test.loc[[attack_index]]
actual_labels = y_test.loc[[attack_index]]

# Preprocess the request using the saved preprocessor
attack_processed = preprocessor.transform(attack_sample)

# Ask the trained model to predict
attack_probabilities = model.predict_proba(attack_processed)
attack_predictions = (attack_probabilities >= 0.5).astype(int)

# Display the original HTTP request
print("ACTUAL HTTP REQUEST:")
display(attack_sample)

# Create a readable prediction table
attack_results = pd.DataFrame({
    "Attack Label": y_test.columns,
    "Probability (%)": (attack_probabilities[0] * 100).round(4),
    "Predicted": attack_predictions[0],
    "Actual": actual_labels.iloc[0].values
})

attack_results["Correct"] = (
    attack_results["Predicted"] == attack_results["Actual"]
)

print("\nMODEL DETECTION RESULTS:")
display(attack_results)

# Show only the predicted and actual attack names
print(
    "Predicted labels:",
    attack_results.loc[
        attack_results["Predicted"] == 1, "Attack Label"
    ].tolist()
)

print(
    "Actual labels:",
    attack_results.loc[
        attack_results["Actual"] == 1, "Attack Label"
    ].tolist()
)

print(
    "All 13 labels matched:",
    bool(attack_results["Correct"].all())
)

print("\nStep 7 completed!")

ACTUAL HTTP REQUEST:

,request_http_request,request_body,request_http_method,request_http_protocol,request_host,request_user_agent,request_referer,request_origin,request_cookie,request_content_type,request_accept,request_accept_language,request_accept_encoding,request_connection
443732,/blog/index.php/2020/03/22%29+UNION+ALL+SELECT...,,GET,HTTP/1.1,test-site.com,Mozilla/5.0 (X11; Linux x86_64; rv:70.0) Gecko...,http://test-site.com/blog/index.php/2020/03/22,,comment_author_url_1aefbe2f76edd740f8e362f39da...,,,,,Close



MODEL DETECTION RESULTS:


,Attack Label,Probability (%),Predicted,Actual,Correct
0,000 - Normal,0.0112,0,0,True
1,272 - Protocol Manipulation,0.0046,0,0,True
2,242 - Code Injection,0.0006,0,0,True
3,88 - OS Command Injection,0.0003,0,0,True
4,126 - Path Traversal,0.0008,0,0,True
5,66 - SQL Injection,99.9843,1,1,True
6,16 - Dictionary-based Password Attack,0.0004,0,0,True
7,310 - Scanning for Vulnerable Software,0.0002,0,0,True
8,153 - Input Data Manipulation,0.0010,0,0,True
9,274 - HTTP Verb Tampering,0.0006,0,0,True


Predicted labels: ['66 - SQL Injection']
Actual labels: ['66 - SQL Injection']
All 13 labels matched: True

Step 7 completed!


In [8]:
# Step 8: Save the SQL Injection demonstration results

output_file = DEMO_DIR / "demo_sql_injection_predictions.csv"

attack_results.to_csv(output_file, index=False)

print("SQL Injection demo results saved successfully!")
print("Saved location:", output_file)
print("Step 8 completed!")

SQL Injection demo results saved successfully!
Saved location: C:\Users\Meishaaa\OneDrive - Sri Lanka Institute of Information Technology\Desktop updates\Desktop\RA 1\Demo outputs\demo_sql_injection_predictions.csv
Step 8 completed!


In [9]:
# Step 9: PP1 Live Detection Demonstration Summary

summary = pd.DataFrame([
    {
        "Test Case": "Normal HTTP Request",
        "Predicted Labels": ", ".join(predicted_attacks),
        "Actual Labels": ", ".join(actual_attacks),
        "All Labels Matched": bool(results_table["Correct"].all())
    },
    {
        "Test Case": "Attack HTTP Request",
        "Predicted Labels": ", ".join(
            attack_results.loc[
                attack_results["Predicted"] == 1, "Attack Label"
            ]
        ),
        "Actual Labels": ", ".join(
            attack_results.loc[
                attack_results["Actual"] == 1, "Attack Label"
            ]
        ),
        "All Labels Matched": bool(attack_results["Correct"].all())
    }
])

display(summary)

summary_file = DEMO_DIR / "demo_summary.csv"
summary.to_csv(summary_file, index=False)

print("Demo summary saved successfully!")
print("Saved location:", summary_file)
print("Step 9 completed!")

,Test Case,Predicted Labels,Actual Labels,All Labels Matched
0,Normal HTTP Request,000 - Normal,000 - Normal,True
1,Attack HTTP Request,66 - SQL Injection,66 - SQL Injection,True


Demo summary saved successfully!
Saved location: C:\Users\Meishaaa\OneDrive - Sri Lanka Institute of Information Technology\Desktop updates\Desktop\RA 1\Demo outputs\demo_summary.csv
Step 9 completed!


In [10]:
# Step 10: Calculate exact-match accuracy

from sklearn.metrics import accuracy_score

# Transform all test requests
X_test_processed = preprocessor.transform(X_test)

# Predict all 13 labels for every test request
all_predictions = model.predict(X_test_processed)

# Compare predictions against the actual labels
exact_accuracy = accuracy_score(y_test, all_predictions)

print("Exact-match accuracy:", round(exact_accuracy * 100, 2), "%")
print("Step 10 completed!")

Exact-match accuracy: 98.21 %
Step 10 completed!


In [11]:
# Step 11: Multi-label demonstration
# One HTTP request can carry MORE THAN ONE attack type at the same time.
# This is why we use One-vs-Rest: one yes/no classifier per attack label.

import numpy as np

actual_matrix = y_test.values
label_counts = actual_matrix.sum(axis=1)

# Find test requests that have 2 or more actual labels
multi_rows = np.where(label_counts >= 2)[0]

# Count how many of them the model got completely right (all 13 labels)
fully_correct = [
    i for i in multi_rows
    if (all_predictions[i] == actual_matrix[i]).all()
]

print("Test requests with 2+ attack labels:", len(multi_rows))
print("Of those, all 13 labels predicted correctly:", len(fully_correct))
print(
    "Multi-label exact-match rate:",
    round(len(fully_correct) / len(multi_rows) * 100, 2), "%"
)

# Show the first request that has 3 labels and was fully correct
three_label = [i for i in fully_correct if label_counts[i] == 3]
multi_position = three_label[0] if three_label else fully_correct[0]

multi_sample = X_test.iloc[[multi_position]]
multi_probabilities = model.predict_proba(preprocessor.transform(multi_sample))

multi_results = pd.DataFrame({
    "Attack Label": y_test.columns,
    "Probability (%)": (multi_probabilities[0] * 100).round(4),
    "Predicted": (multi_probabilities[0] >= 0.5).astype(int),
    "Actual": actual_matrix[multi_position]
})
multi_results["Correct"] = multi_results["Predicted"] == multi_results["Actual"]

print("\nACTUAL HTTP REQUEST:")
print("Method:", multi_sample["request_http_method"].iloc[0])
print("URI   :", multi_sample["request_http_request"].iloc[0][:200])
print("Body  :", str(multi_sample["request_body"].iloc[0])[:200])

print("\nMODEL DETECTION RESULTS:")
display(multi_results)

print("Predicted labels:", multi_results.loc[multi_results["Predicted"] == 1, "Attack Label"].tolist())
print("Actual labels:", multi_results.loc[multi_results["Actual"] == 1, "Attack Label"].tolist())
print("All 13 labels matched:", bool(multi_results["Correct"].all()))

multi_results.to_csv(DEMO_DIR / "demo_multilabel_predictions.csv", index=False)
print("\nStep 11 completed!")

Test requests with 2+ attack labels: 1852
Of those, all 13 labels predicted correctly: 1290
Multi-label exact-match rate: 69.65 %

ACTUAL HTTP REQUEST:
Method: POST
URI   : /blog/wp-includes/js/mediaelement/mediaelement-and-player.min.js?-d+allow_url_include%3d1+-d+auto_prepend_file%3dphp://input
Body  : <?php exec('echo jrpkjls7ygpnhzkjcebd',$colm);echo join("

MODEL DETECTION RESULTS:


,Attack Label,Probability (%),Predicted,Actual,Correct
0,000 - Normal,0.1520,0,0,True
1,272 - Protocol Manipulation,0.0207,0,0,True
2,242 - Code Injection,99.0152,1,1,True
3,88 - OS Command Injection,99.9398,1,1,True
4,126 - Path Traversal,0.0044,0,0,True
5,66 - SQL Injection,3.3953,0,0,True
6,16 - Dictionary-based Password Attack,0.0448,0,0,True
7,310 - Scanning for Vulnerable Software,0.0013,0,0,True
8,153 - Input Data Manipulation,0.0013,0,0,True
9,274 - HTTP Verb Tampering,0.0092,0,0,True


Predicted labels: ['242 - Code Injection', '88 - OS Command Injection', '33 - HTTP Request Smuggling']
Actual labels: ['242 - Code Injection', '88 - OS Command Injection', '33 - HTTP Request Smuggling']
All 13 labels matched: True

Step 11 completed!


In [12]:
# Step 12: Honest limitation - a rare attack the model MISSES
# HTTP Request Smuggling has only 849 training examples (0.12% of the data).
# Recall for this label is 48.10%, so about half are missed at the 0.5 threshold.

smuggling = "33 - HTTP Request Smuggling"
smuggling_col = list(y_test.columns).index(smuggling)

missed_rows = np.where(
    (actual_matrix[:, smuggling_col] == 1) &
    (all_predictions[:, smuggling_col] == 0)
)[0]

print("Smuggling requests in test set:", int(actual_matrix[:, smuggling_col].sum()))
print("Missed by the model:", len(missed_rows))

miss_position = missed_rows[0]
miss_sample = X_test.iloc[[miss_position]]
miss_probabilities = model.predict_proba(preprocessor.transform(miss_sample))

miss_results = pd.DataFrame({
    "Attack Label": y_test.columns,
    "Probability (%)": (miss_probabilities[0] * 100).round(4),
    "Predicted": (miss_probabilities[0] >= 0.5).astype(int),
    "Actual": actual_matrix[miss_position]
})

print("\nACTUAL HTTP REQUEST:")
print("Method:", miss_sample["request_http_method"].iloc[0])
print("URI   :", miss_sample["request_http_request"].iloc[0][:200])

print("\nMODEL DETECTION RESULTS:")
display(miss_results)

print(
    f"\nSmuggling probability = {miss_results.loc[smuggling_col, 'Probability (%)']}% "
    "(below the 50% threshold, so it was not flagged)."
)
print("Why: very few training examples, and its payload looks like HTTP Response Splitting.")
print("Next step: tune a separate threshold per label on a validation set.")

miss_results.to_csv(DEMO_DIR / "demo_missed_smuggling.csv", index=False)
print("\nStep 12 completed!")

Smuggling requests in test set: 210
Missed by the model: 109

ACTUAL HTTP REQUEST:
Method: GET
URI   : /blog/wp-content/plugins/user-registration/assets/js/frontend/my-account.min.js?ver=any%3F%0D%0ASet-cookie%3A+Tamper%3Dae141467-ab17-4349-8101-33296424e82f

MODEL DETECTION RESULTS:


,Attack Label,Probability (%),Predicted,Actual
0,000 - Normal,0.3499,0,0
1,272 - Protocol Manipulation,0.1392,0,0
2,242 - Code Injection,0.0237,0,0
3,88 - OS Command Injection,0.0589,0,0
4,126 - Path Traversal,0.0615,0,0
5,66 - SQL Injection,0.1482,0,0
6,16 - Dictionary-based Password Attack,0.0002,0,0
7,310 - Scanning for Vulnerable Software,0.0006,0,0
8,153 - Input Data Manipulation,0.0008,0,0
9,274 - HTTP Verb Tampering,0.0008,0,0



Smuggling probability = 9.0948% (below the 50% threshold, so it was not flagged).
Why: very few training examples, and its payload looks like HTTP Response Splitting.
Next step: tune a separate threshold per label on a validation set.

Step 12 completed!


In [13]:
# Step 13: Type your own HTTP request
# Web server logs store URLs percent-encoded (e.g. ' becomes %27, / becomes %2F),
# and the SR-BH 2020 dataset is stored the same way. So we encode the typed
# request first, exactly like a browser/log would, before giving it to the model.
# All other headers are left blank so ONLY the request itself is judged.

from urllib.parse import quote, quote_plus

def encode_uri(uri):
    path, has_query, query = uri.partition("?")
    path = quote(path, safe="/")
    if not has_query:
        return path
    encoded_pairs = []
    for pair in query.split("&"):
        key, has_equals, value = pair.partition("=")
        encoded_pairs.append(quote_plus(key) + has_equals + quote_plus(value))
    return path + "?" + "&".join(encoded_pairs)

def detect_request(uri, method="GET", body=""):
    request = {column: "" for column in X_test.columns}
    request["request_http_method"] = method
    request["request_http_protocol"] = "HTTP/1.1"
    request["request_host"] = "test-site.com"
    request["request_http_request"] = encode_uri(uri)
    request["request_body"] = body

    request_df = pd.DataFrame([request])[X_test.columns]
    request_probabilities = model.predict_proba(preprocessor.transform(request_df))[0]

    detected = [
        f"{label} ({probability * 100:.1f}%)"
        for label, probability in zip(y_test.columns, request_probabilities)
        if probability >= 0.5
    ]
    return ", ".join(detected) if detected else "No label above 50%"

demo_requests = [
    ("GET",  "/blog/index.php/2020/03/22/hello-world/", ""),
    ("GET",  "/blog/index.php?p=1", ""),
    ("GET",  "/blog/index.php?id=1' UNION ALL SELECT username,password FROM users-- -", ""),
    ("GET",  "/blog/index.php?id=1) AND SLEEP(5)#", ""),
    ("GET",  "/blog/index.php?page=../../../../etc/passwd", ""),
    ("GET",  "/blog/index.php?q=${@print(chr(122).chr(97))}", ""),
    ("GET",  "/blog/index.php?url=any\r\nSet-cookie: Tamper=1", ""),
    ("GET",  "/blog/index.php?ver=1';cat /etc/passwd", ""),
]

typed_results = pd.DataFrame([
    {"Method": method, "Typed Request": uri, "Model Detection": detect_request(uri, method, body)}
    for method, uri, body in demo_requests
])

pd.set_option("display.max_colwidth", 90)
display(typed_results)

typed_results.to_csv(DEMO_DIR / "demo_typed_requests.csv", index=False)
print("Note: the last one (OS Command Injection) is missed - this label has 64% recall.")
print("Step 13 completed!")

,Method,Typed Request,Model Detection
0,GET,/blog/index.php/2020/03/22/hello-world/,000 - Normal (68.0%)
1,GET,/blog/index.php?p=1,000 - Normal (97.3%)
2,GET,"/blog/index.php?id=1' UNION ALL SELECT username,password FROM users-- -",66 - SQL Injection (84.6%)
3,GET,/blog/index.php?id=1) AND SLEEP(5)#,66 - SQL Injection (99.5%)
4,GET,/blog/index.php?page=../../../../etc/passwd,126 - Path Traversal (80.6%)
5,GET,/blog/index.php?q=${@print(chr(122).chr(97))},242 - Code Injection (99.8%)
6,GET,/blog/index.php?url=any\r\nSet-cookie: Tamper=1,34 - HTTP Response Splitting (100.0%)
7,GET,/blog/index.php?ver=1';cat /etc/passwd,000 - Normal (100.0%)


Note: the last one (OS Command Injection) is missed - this label has 64% recall.
Step 13 completed!


In [14]:
# Step 14: Live input - change the request below and run this cell again

my_method = "GET"
my_uri = "/blog/index.php?id=5 OR 1=1"
my_body = ""

print("Request :", my_method, my_uri)
print("Encoded :", encode_uri(my_uri))
print("Detected:", detect_request(my_uri, my_method, my_body))

Request : GET /blog/index.php?id=5 OR 1=1
Encoded : /blog/index.php?id=5+OR+1%3D1
Detected: 66 - SQL Injection (99.9%)
